# Baseline Model

Train first-pass Logistic Regression, Decision Tree, and Random Forest classifiers for SLA breach prediction using the approved feature dataset.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.models import evaluate_model, train_baseline

importlib.reload(evaluate_model)
importlib.reload(train_baseline)

from src.models.evaluate_model import evaluate_classifier, metrics_to_frame
from src.models.train_baseline import (
    DATA_PATH,
    MODEL_DIR,
    REPORT_PATH,
    build_preprocessor,
    generate_markdown_report,
    load_feature_dataset,
    save_models,
    save_report,
    separate_features_and_target,
    split_train_test,
    train_models,
)


## Load Feature Dataset

In [ ]:
df = load_feature_dataset(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df["sla_breached"].value_counts(normalize=True).rename("share").to_frame().join(
    df["sla_breached"].value_counts().rename("count")
)

## Separate Features And Target

In [ ]:
X, y = separate_features_and_target(df)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
X.head()

## Stratified Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(X, y)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Train breach rate: {y_train.mean():.2%}")
print(f"Test breach rate: {y_test.mean():.2%}")

## Train Baseline Models

In [ ]:
preprocessor = build_preprocessor(X_train)
models = train_models(X_train, y_train, preprocessor)
list(models.keys())

## Evaluate Models

In [ ]:
metrics = [
    evaluate_classifier(model_name, model, X_test, y_test)
    for model_name, model in models.items()
]
comparison = metrics_to_frame(metrics)
comparison

## Save Models And Report

In [ ]:
save_models(models, MODEL_DIR)
report = generate_markdown_report(comparison, metrics, y_train, y_test)
save_report(report, REPORT_PATH)

print(f"Saved models to: {MODEL_DIR}")
print(f"Saved report to: {REPORT_PATH}")